# Draw & Guess — обучение CNN на GPU (Colab)

Большая модель + аугментация. Результат полностью совместим с приложением:
вход 28x28x1 (белые штрихи на чёрном, 0..1), выход softmax по 321 классу,
порядок меток зафиксирован.

**Перед запуском:** Runtime → Change runtime type → Hardware accelerator = **GPU (T4)** → Save.
Затем Runtime → **Run all**. В конце скачается `model.zip`.

In [ ]:
# 1. Проверка GPU
import tensorflow as tf
print('TF', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus if gpus else 'НЕТ — включи Runtime → Change runtime type → GPU')

In [ ]:
# 2. Список классов (порядок ОБЯЗАН совпадать с приложением — не меняй!)
import json
CLASSES = json.loads('["aircraft carrier","airplane","alarm clock","ambulance","angel","ant","anvil","apple","asparagus","axe","backpack","banana","bandage","barn","baseball","baseball bat","basket","basketball","bat","bathtub","beach","bear","beard","bed","bee","belt","bench","bicycle","binoculars","bird","birthday cake","blackberry","blueberry","book","boomerang","bowtie","bracelet","brain","bread","bridge","broccoli","broom","bucket","bulldozer","bus","bush","butterfly","cactus","cake","calculator","calendar","camel","camera","campfire","candle","cannon","canoe","car","carrot","castle","cat","ceiling fan","cello","cell phone","chair","chandelier","church","circle","clarinet","clock","cloud","coffee cup","compass","computer","cookie","couch","cow","crab","crayon","crocodile","crown","cruise ship","cup","diamond","dishwasher","dog","dolphin","donut","door","dragon","dresser","drill","drums","duck","dumbbell","ear","elephant","envelope","eraser","eye","eyeglasses","face","fan","feather","fence","finger","fire hydrant","fireplace","firetruck","fish","flamingo","flashlight","flip flops","floor lamp","flower","flying saucer","foot","fork","frog","frying pan","garden","garden hose","giraffe","golf club","grapes","grass","guitar","hamburger","hammer","hand","harp","hat","headphones","hedgehog","helicopter","helmet","hockey puck","hockey stick","horse","hospital","hot air balloon","hot dog","hot tub","hourglass","house","house plant","ice cream","jacket","jail","kangaroo","key","keyboard","knife","ladder","lantern","laptop","leaf","leg","light bulb","lighter","lighthouse","lightning","lion","lipstick","lobster","lollipop","mailbox","map","marker","matches","megaphone","mermaid","microphone","microwave","monkey","moon","mosquito","motorbike","mountain","mouse","moustache","mouth","mug","mushroom","nail","necklace","nose","octopus","onion","oven","owl","paintbrush","paint can","palm tree","panda","pants","paper clip","parachute","parrot","passport","peanut","pear","peas","pencil","penguin","piano","pickup truck","picture frame","pig","pillow","pineapple","pizza","pliers","police car","pool","popsicle","postcard","potato","power outlet","purse","rabbit","raccoon","radio","rain","rainbow","rake","remote control","rhinoceros","rifle","river","roller coaster","rollerskates","sailboat","sandwich","saw","saxophone","school bus","scissors","scorpion","screwdriver","sea turtle","see saw","shark","sheep","shoe","shorts","shovel","sink","skateboard","skull","skyscraper","sleeping bag","smiley face","snail","snake","snowflake","snowman","soccer ball","sock","speedboat","spider","spoon","square","squirrel","stairs","star","steak","stereo","stethoscope","stop sign","stove","strawberry","streetlight","string bean","submarine","suitcase","sun","swan","sweater","swing set","sword","syringe","table","teapot","teddy-bear","telephone","television","tennis racquet","tent","The Eiffel Tower","tiger","toaster","toilet","tooth","toothbrush","toothpaste","tornado","tractor","traffic light","train","tree","triangle","trombone","truck","trumpet","t-shirt","umbrella","underwear","van","vase","violin","washing machine","watermelon","waterslide","whale","wheel","windmill","wine bottle","wine glass","wristwatch","zebra"]')
print(len(CLASSES), 'classes')

# Сколько примеров на класс качать (умеренно — чтобы влезть в бесплатный Colab по ОЗУ)
PER_CLASS = 7000
N_TRAIN, N_VAL, N_TEST = 6000, 500, 500
ROW = 784

In [ ]:
# 3. Скачивание данных QuickDraw (ranged — только нужные строки, не целые файлы)
import urllib.parse, requests, numpy as np
BUCKET = 'https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/'

def npy_header(head):
    assert head[:6] == b'\x93NUMPY'
    if head[6] == 1:
        hlen = int.from_bytes(head[8:10],'little'); start = 10
    else:
        hlen = int.from_bytes(head[8:12],'little'); start = 12
    d = eval(head[start:start+hlen].decode('latin1'), {'__builtins__':{}}, {})
    return start+hlen, d['shape'][0]

def fetch(name, k):
    url = BUCKET + urllib.parse.quote(name) + '.npy'
    h = requests.get(url, headers={'Range':'bytes=0-255'}, timeout=60); h.raise_for_status()
    off, total = npy_header(h.content); k = min(k, total)
    r = requests.get(url, headers={'Range': f'bytes={off}-{off+k*ROW-1}'}, timeout=300); r.raise_for_status()
    a = np.frombuffer(r.content, dtype=np.uint8)
    return a[:(len(a)//ROW)*ROW].reshape(-1, ROW)

Xtr=[];ytr=[];Xva=[];yva=[];Xte=[];yte=[]
for i,name in enumerate(CLASSES):
    a = fetch(name, PER_CLASS); n=len(a)
    ntr=min(N_TRAIN, n-N_VAL-N_TEST); b=ntr+N_VAL
    Xtr.append(a[:ntr]); ytr += [i]*ntr
    Xva.append(a[ntr:b]); yva += [i]*(b-ntr)
    Xte.append(a[b:b+N_TEST]); yte += [i]*len(a[b:b+N_TEST])
    if (i+1)%40==0: print(f'{i+1}/{len(CLASSES)}')
Xtr=np.concatenate(Xtr); ytr=np.array(ytr,dtype=np.int16)
Xva=np.concatenate(Xva); yva=np.array(yva,dtype=np.int16)
Xte=np.concatenate(Xte); yte=np.array(yte,dtype=np.int16)
print('train', Xtr.shape, 'val', Xva.shape, 'test', Xte.shape)

In [ ]:
# 4. Большая модель + аугментация, обучение на GPU
# Память: данные держим как uint8, нормализуем /255 ПОБАТЧНО через tf.data
# (float32-копии всего набора нет — иначе Colab падает по ОЗУ).
# Корректность: ОДИН раз глобально перемешиваем массив (данные лежат блоками
# по классам) — тогда батчи мультиклассовые и BatchNorm/валидация работают.
from tensorflow.keras import layers, models, callbacks
import numpy as np
NUM = len(CLASSES); AUTOTUNE = tf.data.AUTOTUNE

p = np.random.permutation(len(Xtr))   # глобальный shuffle (один проход)
Xtr = Xtr[p]; ytr = ytr[p]
print('shuffled train', Xtr.shape, Xtr.dtype)

def make_ds(X, y, training):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(50000)   # данные уже перемешаны → буфер реально мешает классы
    ds = ds.batch(512)
    ds = ds.map(lambda xb, yb: (tf.cast(tf.reshape(xb, (-1, 28, 28, 1)), tf.float32) / 255.0, yb),
                num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_ds(Xtr, ytr, True)
val_ds   = make_ds(Xva, yva, False)

aug = models.Sequential([
    layers.RandomRotation(0.08, fill_mode='constant'),
    layers.RandomTranslation(0.1, 0.1, fill_mode='constant'),
    layers.RandomZoom(0.1, fill_mode='constant'),
], name='augment')

def block(x, f):
    x = layers.Conv2D(f, 3, padding='same', activation='relu')(x); x = layers.BatchNormalization()(x)
    x = layers.Conv2D(f, 3, padding='same', activation='relu')(x); x = layers.BatchNormalization()(x)
    x = layers.MaxPool2D()(x); x = layers.Dropout(0.25)(x); return x

# Вход модели — нормализованные [0,1] изображения (как присылает браузер).
inp = layers.Input((28, 28, 1))
x = aug(inp)
x = block(x, 64); x = block(x, 128)
x = layers.Conv2D(256, 3, padding='same', activation='relu')(x); x = layers.BatchNormalization()(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='relu')(x); x = layers.Dropout(0.4)(x)
out = layers.Dense(NUM, activation='softmax')(x)
model = models.Model(inp, out)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3')])
model.summary()

cbs = [callbacks.EarlyStopping(monitor='val_top3', mode='max', patience=4, restore_best_weights=True),
       callbacks.ReduceLROnPlateau(monitor='val_top3', mode='max', factor=0.5, patience=2)]
model.fit(train_ds, validation_data=val_ds, epochs=40, callbacks=cbs, verbose=2)

In [ ]:
# 5. Оценка на held-out тесте
import numpy as np
test_ds = make_ds(Xte, yte, False)
probs = model.predict(test_ds, verbose=0)
top1 = (probs.argmax(1) == yte).mean()
top3 = np.mean([yte[i] in probs[i].argsort()[-3:] for i in range(len(yte))])
print(f'TEST top1={top1*100:.1f}%  top3={top3*100:.1f}%')

In [ ]:
# 6. Экспорт в TF.js (drop-in для приложения) + скачивание model.zip
import json, os
model.export('/content/saved_model')
!pip -q install tensorflowjs
!tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model /content/saved_model /content/model_out
json.dump(CLASSES, open('/content/model_out/labels.json','w'), ensure_ascii=False)
print('files:', os.listdir('/content/model_out'))
!cd /content/model_out && zip -r /content/model.zip . >/dev/null
from google.colab import files
files.download('/content/model.zip')
print('Готово — скачивается model.zip')